# 01b — The Curse of Dimensionality

## The One-Sentence Version
As you add dimensions, space gets so incomprehensibly vast that your data becomes sparse, distances become meaningless, and algorithms that worked beautifully in low dimensions quietly fall apart.

## What You'll Build Intuition For
- Why high-dimensional spaces are almost entirely empty
- Why "nearby" loses meaning when dimensions grow
- Why models need exponentially more data as dimensions increase
- Why this isn't just theory — it breaks real systems

## Prerequisites
- [01a — Building Intuition](01a_building_intuition.ipynb) — what a dimension is

## The Story

In 01a we learned that a dimension is just a question. More questions, more dimensions. Simple.

But here's the thing nobody warns you about: adding dimensions doesn't just make the space *bigger*. It makes it **hostile**.

Imagine you live in a 1D world — a line from 0 to 1. You scatter 10 points along it, and they're reasonably well-spaced. Now move to a 2D square. Those same 10 points rattle around in a space that's 1×1 — it feels emptier. In a 3D cube, even emptier.

By the time you hit 10 dimensions, those 10 points are so lost in the vast emptiness that each one might as well be alone. The space between them is incomprehensible. And it gets worse: distances stop working. In very high dimensions, the closest point and the farthest point are almost the same distance away. "Near" and "far" collapse into a meaningless blur.

This isn't abstract mathematics. This is why your k-nearest-neighbours model degrades when you add features. This is why your clustering produces garbage on wide datasets. This is *the* reason dimensionality reduction exists.

Let's see it happen.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import gamma
from scipy.spatial.distance import cdist

from utils.plotting import apply_style, COLOURS
from utils.data_generators import make_low_d_in_high_d, make_two_class_with_noise

apply_style()
rng = np.random.default_rng(42)

## The Volume Explosion

Think about a circle inscribed inside a square. In 2D, the circle fills about 79% of the square. In 3D, a sphere inscribed in a cube fills about 52%. What happens as dimensions climb?

In [ ]:
# Volume of a hypersphere inscribed in a unit hypercube (radius = 0.5)
def hypersphere_volume(d, r=0.5):
    """Volume of a d-dimensional sphere of radius r."""
    return (np.pi ** (d / 2) / gamma(d / 2 + 1)) * r ** d

dims = range(1, 51)
ratios = [hypersphere_volume(d) / 1.0 for d in dims]  # cube has volume 1

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(dims, ratios, color=COLOURS["noise"], alpha=0.8, edgecolor="white", linewidth=0.3)
ax.set_xlabel("Number of Dimensions")
ax.set_ylabel("Fraction of Hypercube Filled")
ax.set_title("The Sphere Vanishes")
ax.set_yscale("log")
ax.set_ylim(bottom=1e-35)

# Annotate a few key values
for d_anno in [2, 3, 10, 20]:
    val = hypersphere_volume(d_anno)
    ax.annotate(f"d={d_anno}: {val:.2e}", (d_anno, val),
                textcoords="offset points", xytext=(10, 5),
                fontsize=9, color=COLOURS["neutral"])

plt.tight_layout()
plt.savefig("visuals/01b_sphere_vanishes.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"2D circle fills {hypersphere_volume(2):.1%} of its square")
print(f"3D sphere fills {hypersphere_volume(3):.1%} of its cube")
print(f"10D hypersphere fills {hypersphere_volume(10):.6%} of its hypercube")
print(f"50D? {hypersphere_volume(50):.2e} — effectively zero")

> **Key insight:** In high dimensions, almost all the volume is in the corners. The "middle" of the space — where a sphere sits — is essentially empty. Your 2D/3D intuition about shapes completely betrays you.

<details>
<summary><b>The Maths Behind This</b></summary>

The volume of a $d$-dimensional hypersphere of radius $r$ is:

$$V_d(r) = \frac{\pi^{d/2}}{\Gamma(d/2 + 1)} \cdot r^d$$

For a sphere inscribed in the unit hypercube, $r = 0.5$, and the hypercube has volume 1. The ratio $V_d(0.5) / 1$ drops super-exponentially because the $r^d = 0.5^d$ term dominates, and the gamma function in the denominator grows factorially.

As $d \to \infty$, this ratio $\to 0$ — the sphere becomes an infinitesimal fraction of its bounding cube.

</details>

## Everything Is on the Edge

Here's another way the curse bites. Consider a unit hypercube $[0, 1]^d$. How much of its volume is "interior" — more than 1% from any wall?

In 2D, most of the square is interior. In high dimensions, virtually *everything* is pressed against the boundary.

In [ ]:
# What fraction of a hypercube's volume is "interior" (more than ε from any wall)?
# Interior has side length (1 - 2ε), so its fraction is (1 - 2ε)^d

dims = np.arange(1, 501)
shell_thickness = 0.01  # 1% from the boundary
interior_fraction = (1 - 2 * shell_thickness) ** dims

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(dims, interior_fraction, color=COLOURS["noise"], linewidth=2)
ax.axhline(y=0.01, color=COLOURS["neutral"], linestyle="--", alpha=0.5, label="1% interior")
ax.fill_between(dims, interior_fraction, alpha=0.15, color=COLOURS["noise"])
ax.set_xlabel("Dimensions")
ax.set_ylabel("Fraction of Volume in Interior")
ax.set_title("In High Dimensions, Everything Lives on the Boundary")
ax.legend()

# Mark some key crossings
for d_mark, label in [(100, "d=100"), (230, "d=230")]:
    frac = (1 - 2 * shell_thickness) ** d_mark
    ax.annotate(f"{label}: {frac:.2%}", (d_mark, frac),
                textcoords="offset points", xytext=(15, 10),
                fontsize=9, color=COLOURS["neutral"],
                arrowprops=dict(arrowstyle="->", color=COLOURS["neutral"]))

plt.tight_layout()
plt.savefig("visuals/01b_boundary_concentration.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"d=2:   {(0.98)**2:.1%} interior — most of the square")
print(f"d=10:  {(0.98)**10:.1%} interior — still okay")
print(f"d=100: {(0.98)**100:.1%} interior — 87% is in the thin outer shell")
print(f"d=500: {(0.98)**500:.6%} interior — essentially everything is on the edge")

> **Key insight:** There is no "interior" in high-dimensional space. All your data points are pressed against the walls. Algorithms that rely on local density (like k-NN) break because there's no meaningful density structure left.

## Distance Concentration — The Killer

This is the most practically destructive consequence of the curse. In high dimensions, the distances from any reference point to all other points become almost identical. The nearest neighbour and farthest neighbour are nearly the same distance away.

Watch the distance histograms tighten into spikes as dimensions increase.

In [ ]:
# Distance distributions from one reference point to 199 others, at increasing dimensions

dims_to_test = [2, 5, 10, 50, 100, 500, 1000, 5000]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, d in zip(axes.flat, dims_to_test):
    points = rng.uniform(0, 1, size=(200, d))
    dists = cdist(points[:1], points[1:], metric="euclidean")[0]
    dists_normalised = dists / np.sqrt(d)  # normalise for visual comparison

    ax.hist(dists_normalised, bins=30, alpha=0.7,
            color=COLOURS["noise"], edgecolor="white", linewidth=0.5)
    ax.set_title(f"d = {d}")

    contrast = (dists.max() - dists.min()) / (dists.min() + 1e-10)
    ax.text(0.95, 0.9, f"contrast: {contrast:.2f}",
            transform=ax.transAxes, fontsize=9, ha="right",
            color=COLOURS["neutral"])

fig.suptitle("Distance Distributions Collapse in High Dimensions", fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/01b_distance_concentration.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Contrast ratio (max - min) / min as a single line plot

dims_range = [2, 5, 10, 20, 50, 100, 200, 500, 1000, 2000, 5000]
contrasts = []
for d in dims_range:
    pts = rng.uniform(0, 1, size=(200, d))
    dists = np.linalg.norm(pts[1:] - pts[0], axis=1)
    contrasts.append((dists.max() - dists.min()) / dists.min())

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(dims_range, contrasts, "o-", color=COLOURS["noise"], markersize=8, linewidth=2)
ax.set_xlabel("Dimensions")
ax.set_ylabel("(max_dist − min_dist) / min_dist")
ax.set_title("Distance Contrast Ratio → 0")
ax.set_xscale("log")

plt.tight_layout()
plt.savefig("visuals/01b_contrast_ratio.png", dpi=150, bbox_inches="tight")
plt.show()

> **Key insight:** When all distances are approximately equal, "nearest neighbour" is arbitrary. k-NN breaks. Clustering breaks. Any algorithm relying on "close vs far" is compromised. Those extra dimensions aren't just adding information — they're actively degrading your distance metrics.

<details>
<summary><b>The Maths Behind This</b></summary>

For uniform random points in $[0,1]^d$, the squared Euclidean distance from a reference point to another is:

$$\|x - y\|^2 = \sum_{i=1}^{d} (x_i - y_i)^2$$

Each $(x_i - y_i)^2$ is an independent random variable with some mean $\mu$ and variance $\sigma^2$. By the law of large numbers, as $d \to \infty$:

$$\frac{\|x - y\|^2}{d} \to \mu$$

All squared distances concentrate around $d\mu$, so:

$$\frac{d_{\max} - d_{\min}}{d_{\min}} \to 0$$

Intuitively: each dimension adds a small random contribution to the distance. With enough dimensions, these average out and every point ends up at roughly the same distance from every other point.

</details>

## The Data Hunger Problem

To maintain the same *density* of data as dimensions grow, you need exponentially more points. If you want 10 points per unit length in each dimension, the total needed is $10^d$.

This isn't a slow growth. It's a wall.

In [ ]:
# Points needed to maintain 10 samples per unit in each dimension

density = 10  # points per unit length per dimension
dims = np.arange(1, 21)
points_needed = density ** dims.astype(float)

fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(dims, points_needed, "o-", color=COLOURS["signal"], markersize=8, linewidth=2)
ax.axhline(y=1e6, color=COLOURS["accent"], linestyle="--", label="1 million points")
ax.axhline(y=1e9, color=COLOURS["noise"], linestyle="--", label="1 billion points")
ax.axhline(y=1e80, color="#8B0000", linestyle="--", alpha=0.6,
           label="Atoms in observable universe (~10⁸⁰)")
ax.set_xlabel("Dimensions")
ax.set_ylabel("Points Needed for Same Coverage")
ax.set_title("The Exponential Wall: Why Brute Force Fails")
ax.legend(loc="upper left")
ax.set_xlim(0.5, 20.5)

plt.tight_layout()
plt.savefig("visuals/01b_exponential_wall.png", dpi=150, bbox_inches="tight")
plt.show()

To put that in perspective:

| Dimensions | Points Needed (10/dim) | Context |
|-----------|----------------------|---------|
| 1 | 10 | Trivial |
| 2 | 100 | A spreadsheet |
| 5 | 100,000 | A medium dataset |
| 10 | 10 billion | Larger than most datasets |
| 20 | 10²⁰ | More than grains of sand on Earth |
| 100 | 10¹⁰⁰ | More than atoms in the universe |

> **Key insight:** You will never have enough data to fill high-dimensional space. Never. This is why dimensionality reduction isn't optional — it's survival.

## Seeing It Break in Practice

Enough theory. Let's build a perfectly good classifier and watch the curse destroy it.

We'll create a simple 2D classification problem (two signal features), then keep adding random noise dimensions. The decision boundary doesn't change — but the algorithm's ability to find it crumbles.

In [ ]:
# k-NN accuracy as noise dimensions are added to a 2D signal
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

# Use our utility to generate the datasets
datasets = make_two_class_with_noise(n=300, seed=42)

total_dims = sorted(datasets.keys())
accuracies = []

for d in total_dims:
    X, y = datasets[d]
    knn = KNeighborsClassifier(n_neighbors=5)
    scores = cross_val_score(knn, X, y, cv=5)
    accuracies.append(scores.mean())

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(total_dims, accuracies, "o-", color=COLOURS["noise"], markersize=8, linewidth=2)
ax.axhline(y=accuracies[0], color=COLOURS["success"], linestyle="--", alpha=0.5,
           label=f"Baseline (2D only): {accuracies[0]:.1%}")
ax.set_xlabel("Total Dimensions (2 signal + noise)")
ax.set_ylabel("k-NN Accuracy (5-fold CV)")
ax.set_title("Adding Noise Dimensions Destroys a Perfectly Good Classifier")
ax.legend()
ax.set_ylim(0.4, 1.0)

plt.tight_layout()
plt.savefig("visuals/01b_knn_degradation.png", dpi=150, bbox_inches="tight")
plt.show()

print("Accuracy by total dimensions:")
for d, acc in zip(total_dims, accuracies):
    noise_d = d - 2
    print(f"  {d:>4}D (2 signal + {noise_d:>3} noise): {acc:.1%}")

> **Key insight:** Those extra dimensions aren't neutral. They actively damage performance. Every dimension you add that doesn't carry signal is sand in the gears.

## The Silver Lining — Why Reduction Works

The curse sounds fatal. But there's a reason this repo doesn't end here.

Real data almost never fills the full ambient space. A dataset with 100 columns rarely has 100 independent directions of variation. Most of those columns are correlated, redundant, or noise. The data actually lives on a much lower-dimensional surface embedded in the high-dimensional space.

Let's prove it.

In [ ]:
# Generate 100D data that secretly lives on a 3D manifold
from sklearn.decomposition import PCA
from utils.plotting import plot_explained_variance

X, Z_true = make_low_d_in_high_d(n=500, intrinsic_d=3, ambient_d=100, seed=42)

print(f"Data shape: {X.shape} — looks like 100 dimensions")
print(f"But it was generated from only {Z_true.shape[1]} underlying factors")

pca = PCA().fit(X)
fig = plot_explained_variance(pca.explained_variance_ratio_[:20],
                              title="100 Dimensions, but Only 3 Matter")
plt.savefig("visuals/01b_silver_lining.png", dpi=150, bbox_inches="tight")
plt.show()

n_95 = np.searchsorted(np.cumsum(pca.explained_variance_ratio_), 0.95) + 1
print(f"\n{n_95} components capture 95% of the variance (out of {X.shape[1]})")

> The curse is severe. But real-world data has structure. A 10,000-dimensional medical dataset doesn't have 10,000 independent axes of variation — it has maybe 10 or 50 real underlying factors, and the rest are correlated echoes and noise.
>
> Finding that structure is what the rest of this repo is about.

## Where This Matters

**Clinical Decision Support:** A hospital's EHR system records 500+ features per patient. A naive similarity search ("find me patients like this one") uses all 500 dimensions — and distance concentration makes the results nearly random. Clinicians who trust these results are being misled by the curse. Dimensionality reduction before similarity search isn't a nice-to-have — it's the difference between useful and dangerous.

**Simulation Design of Experiments:** In operational simulation (FAER-M, wargaming), you might want to explore a 30-dimensional scenario space. Grid-searching with just 5 levels per dimension requires $5^{30} \approx 10^{21}$ runs. Even Latin Hypercube sampling struggles because the points are so sparse relative to the volume. Understanding the curse helps you design smarter experimental plans — and know when you need to reduce dimensions before exploring.

## Feynman Check

Can you explain these without reaching for jargon?

1. **In your own words, why does k-nearest-neighbours struggle in 1000 dimensions?**

2. **If you have 1000 patients each with 500 measurements, why can't you just "let the algorithm figure it out"?**

3. **Explain the distance concentration problem to a colleague without using the word "dimensionality."**

4. **Why is "more data features = better model" dangerously wrong?**

If you can answer these, you understand the *problem*. Now let's ground it in reality with **01c — Real-World High Dimensions**, before we start learning how to fight back in Module 02.